In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# File: FL_IDS_Training.ipynb — Project paths

from pathlib import Path

# Permanent Google Drive location
DRIVE_ROOT = Path("/content/drive/MyDrive/IDSPRESERVING")

# Temporary Colab location for fast training
LOCAL_ROOT = Path("/content/IDSPRESERVING")

# Persistent directories
DRIVE_DATA = DRIVE_ROOT / "data" / "processed"
DRIVE_MODELS = DRIVE_ROOT / "models" / "stage1"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "stage1"

# Temporary local training directories
LOCAL_DATA = LOCAL_ROOT / "data" / "processed"

# Create persistent output directories
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

print("Drive root :", DRIVE_ROOT)
print("Local root :", LOCAL_ROOT)
print("Models     :", DRIVE_MODELS)
print("Results    :", DRIVE_RESULTS)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/IDSPRESERVING")
BASE.mkdir(parents=True, exist_ok=True)

print(BASE)

/content/drive/MyDrive/IDSPRESERVING


In [ ]:
print(BASE)
print(BASE.exists())


/content/drive/MyDrive/IDSPRESERVING
True


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/IDSPRESERVING")

DATA_DIR = BASE / "data" / "processed"

(DATA_DIR / "scaled").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "clients").mkdir(parents=True, exist_ok=True)

print("Folders created:")
print(DATA_DIR / "scaled")
print(DATA_DIR / "clients")

Folders created:
/content/drive/MyDrive/IDSPRESERVING/data/processed/scaled
/content/drive/MyDrive/IDSPRESERVING/data/processed/clients


In [ ]:
from pathlib import Path
import numpy as np

BASE = Path("/content/drive/MyDrive/IDSPRESERVING")
DATA_DIR = BASE / "data" / "processed"

print("Checking files...\n")

# Scaled datasets
for name in [
    "X_train.npy",
    "y_train.npy",
    "X_val.npy",
    "y_val.npy",
    "X_test.npy",
    "y_test.npy"
]:
    path = DATA_DIR / "scaled" / name
    print(f"{name:15} -> {path.exists()}")

print("\nClient partitions:")

for i in range(1, 5):
    path = DATA_DIR / "clients" / f"client_{i}_indices.npy"
    print(f"client_{i}_indices.npy -> {path.exists()}")

print("\nMetadata:")

for name in [
    "feature_names.txt",
    "dataset_info.txt",
    "scaler.joblib"
]:
    path = DATA_DIR / name
    print(f"{name:18} -> {path.exists()}")

Checking files...

X_train.npy     -> True
y_train.npy     -> True
X_val.npy       -> True
y_val.npy       -> True
X_test.npy      -> True
y_test.npy      -> True

Client partitions:
client_1_indices.npy -> True
client_2_indices.npy -> True
client_3_indices.npy -> True
client_4_indices.npy -> True

Metadata:
feature_names.txt  -> True
dataset_info.txt   -> True
scaler.joblib      -> True


In [ ]:
X_train = np.load(
    DATA_DIR / "scaled" / "X_train.npy",
    mmap_mode="r"
)

y_train = np.load(
    DATA_DIR / "scaled" / "y_train.npy",
    mmap_mode="r"
)

X_val = np.load(
    DATA_DIR / "scaled" / "X_val.npy",
    mmap_mode="r"
)

X_test = np.load(
    DATA_DIR / "scaled" / "X_test.npy",
    mmap_mode="r"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (2264594, 78)
y_train: (2264594,)
X_val: (283074, 78)
X_test: (283075, 78)


In [ ]:
!pip install -q "flwr[simulation]" torch pandas numpy scikit-learn matplotlib seaborn joblib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4

In [ ]:
import torch
import flwr as fl

print("PyTorch:", torch.__version__)
print("Flower:", fl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No CUDA GPU detected")

PyTorch: 2.11.0+cu128
Flower: 1.37.0
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
from pathlib import Path

SRC_DIR = BASE / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)

model_code = r'''
import torch
import torch.nn as nn


class IDSModel(nn.Module):

    def __init__(self, input_size=78):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x)
'''

(SRC_DIR / "model.py").write_text(model_code)

print("Created:", SRC_DIR / "model.py")

Created: /content/drive/MyDrive/IDSPRESERVING/src/model.py


In [ ]:
import sys
import torch

sys.path.insert(0, str(SRC_DIR))

from model import IDSModel

model = IDSModel(input_size=78)

print(model)

# Test with 5 samples having 78 features
dummy_input = torch.randn(5, 78).cuda()

model = model.cuda()

output = model(dummy_input)

print("\nInput shape :", dummy_input.shape)
print("Output shape:", output.shape)
print("Device      :", output.device)

IDSModel(
  (network): Sequential(
    (0): Linear(in_features=78, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

Input shape : torch.Size([5, 78])
Output shape: torch.Size([5, 1])
Device      : cuda:0


In [ ]:
import shutil
from pathlib import Path

LOCAL_BASE = Path("/content/IDSPRESERVING")
LOCAL_DATA = LOCAL_BASE / "data" / "processed"

LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print("Copying processed data from Google Drive to Colab runtime...")

Copying processed data from Google Drive to Colab runtime...


In [ ]:
# Copy scaled datasets
drive_scaled = DATA_DIR / "scaled"
local_scaled = LOCAL_DATA / "scaled"
local_scaled.mkdir(parents=True, exist_ok=True)

for file_name in [
    "X_train.npy",
    "y_train.npy",
    "X_val.npy",
    "y_val.npy",
    "X_test.npy",
    "y_test.npy"
]:
    src = drive_scaled / file_name
    dst = local_scaled / file_name

    if not dst.exists():
        print(f"Copying {file_name}...")
        shutil.copy2(src, dst)
    else:
        print(f"{file_name} already exists")

# Copy client partitions
drive_clients = DATA_DIR / "clients"
local_clients = LOCAL_DATA / "clients"
local_clients.mkdir(parents=True, exist_ok=True)

for i in range(1, 5):
    file_name = f"client_{i}_indices.npy"

    src = drive_clients / file_name
    dst = local_clients / file_name

    if not dst.exists():
        print(f"Copying {file_name}...")
        shutil.copy2(src, dst)
    else:
        print(f"{file_name} already exists")

print("\nDataset copy completed.")

Copying X_train.npy...
Copying y_train.npy...
Copying X_val.npy...
Copying y_val.npy...
Copying X_test.npy...
Copying y_test.npy...
Copying client_1_indices.npy...
Copying client_2_indices.npy...
Copying client_3_indices.npy...
Copying client_4_indices.npy...

Dataset copy completed.


In [ ]:
import numpy as np

X_train_local = np.load(
    LOCAL_DATA / "scaled" / "X_train.npy",
    mmap_mode="r"
)

y_train_local = np.load(
    LOCAL_DATA / "scaled" / "y_train.npy",
    mmap_mode="r"
)

print("X_train:", X_train_local.shape)
print("y_train:", y_train_local.shape)

for i in range(1, 5):
    indices = np.load(
        LOCAL_DATA / "clients" / f"client_{i}_indices.npy"
    )

    print(
        f"Client {i}: {len(indices):,} samples"
    )

X_train: (2264594, 78)
y_train: (2264594,)
Client 1: 566,148 samples
Client 2: 566,148 samples
Client 3: 566,148 samples
Client 4: 566,150 samples


In [ ]:
# File: Colab notebook cell — Flower client imports and configuration

import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

import flwr as fl

PROJECT_ROOT = Path("/content/IDSPRESERVING")
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "scaled"
CLIENT_DIR = PROJECT_ROOT / "data" / "processed" / "clients"
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

from model import IDSModel

# Federated learning configuration
NUM_CLIENTS = 4
BATCH_SIZE = 1024
LOCAL_EPOCHS = 1
LEARNING_RATE = 0.001

# Use GPU when available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Flower client configuration loaded")
print("Clients:", NUM_CLIENTS)
print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Local epochs:", LOCAL_EPOCHS)
print("Learning rate:", LEARNING_RATE)

Flower client configuration loaded
Clients: 4
Device: cuda
Batch size: 1024
Local epochs: 1
Learning rate: 0.001


In [ ]:
# File: Colab notebook cell — Memory-efficient client dataset loader

class ClientDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        real_index = self.indices[index]

        x = np.array(
            self.X[real_index],
            dtype=np.float32,
            copy=True
        )

        y = np.float32(self.y[real_index])

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.float32)
        )


def load_client_data(client_id):
    index_path = (
        CLIENT_DIR / f"client_{client_id}_indices.npy"
    )

    # Client indices
    indices = np.load(index_path)

    # Memory-mapped training data
    X_train = np.load(
        DATA_DIR / "X_train.npy",
        mmap_mode="r"
    )

    y_train = np.load(
        DATA_DIR / "y_train.npy",
        mmap_mode="r"
    )

    dataset = ClientDataset(
        X_train,
        y_train,
        indices
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )

    return loader, len(dataset)


# Test only Client 1
client_loader, client_size = load_client_data(1)

print("Client 1 samples:", client_size)
print("Number of batches:", len(client_loader))

Client 1 samples: 566148
Number of batches: 553


In [ ]:
# File: Colab notebook cell — Local client training smoke test

def train_local_model(model, train_loader, epochs=1):
    model.to(DEVICE)
    model.train()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    total_loss = 0.0

    for epoch in range(epochs):
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            logits = model(X_batch).squeeze(1)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        average_loss = epoch_loss / len(train_loader)
        total_loss = average_loss

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {average_loss:.4f}"
        )

    return total_loss


# Smoke test
model = IDSModel(input_size=78)

loss = train_local_model(
    model,
    client_loader,
    epochs=1
)

print("Smoke test completed.")
print("Final loss:", loss)

Epoch 1/1 - Loss: 0.0797
Smoke test completed.
Final loss: 0.0796703065713534


In [ ]:
# File: Colab notebook cell — Make project modules visible to Flower/Ray

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/IDSPRESERVING")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("SRC_DIR:", SRC_DIR)
print("SRC exists:", SRC_DIR.exists())

from model import IDSModel

print("IDSModel import successful.")

SRC_DIR: /content/IDSPRESERVING/src
SRC exists: True
IDSModel import successful.


In [ ]:
# File: Colab notebook cell — IDS model definition for Flower workers

class IDSModel(nn.Module):
    def __init__(self, input_size=78):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


test_model = IDSModel(input_size=78)

print("IDSModel defined successfully.")
print(
    "Parameters:",
    sum(p.numel() for p in test_model.parameters())
)

IDSModel defined successfully.
Parameters: 20481


In [ ]:
# File: Colab notebook cell — Flower federated client

import flwr as fl
import torch
import torch.nn as nn


class FlowerIDSClient(fl.client.NumPyClient):

    def __init__(self, client_id):
        self.client_id = client_id

        # Load this client's data
        self.train_loader, self.num_samples = load_client_data(
            client_id
        )

        # Create a fresh local model
        self.model = IDSModel(input_size=78).to(DEVICE)

    def get_parameters(self, config):
        return [
            val.detach().cpu().numpy()
            for val in self.model.state_dict().values()
        ]

    def set_parameters(self, parameters):
        state_dict = self.model.state_dict()

        new_state_dict = {
            key: torch.tensor(
                value,
                dtype=state_dict[key].dtype
            )
            for key, value in zip(
                state_dict.keys(),
                parameters
            )
        }

        self.model.load_state_dict(new_state_dict)

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        local_epochs = int(
            config.get("local_epochs", 1)
        )

        loss = train_local_model(
            self.model,
            self.train_loader,
            epochs=local_epochs
        )

        return (
            self.get_parameters(config),
            self.num_samples,
            {"loss": float(loss)}
        )


def client_fn(context):
    client_id = int(
        context.node_config["partition-id"]
    ) + 1

    return FlowerIDSClient(client_id).to_client()


print("Flower client definition loaded successfully.")

Flower client definition loaded successfully.


In [ ]:
# File: Colab notebook Cell 20 — FedAvg strategy with final model capture

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load global validation data
X_val = np.load(
    DATA_DIR / "X_val.npy",
    mmap_mode="r"
)

y_val = np.load(
    DATA_DIR / "y_val.npy",
    mmap_mode="r"
)


def evaluate_global_model(
    server_round,
    parameters,
    config
):
    model = IDSModel(input_size=78)

    state_dict = model.state_dict()

    new_state_dict = {
        key: torch.tensor(
            value,
            dtype=state_dict[key].dtype
        )
        for key, value in zip(
            state_dict.keys(),
            parameters
        )
    }

    model.load_state_dict(new_state_dict)
    model.to(DEVICE)
    model.eval()

    predictions = []
    actual = []

    batch_size = 4096

    with torch.no_grad():
        for start in range(0, len(X_val), batch_size):

            end = min(
                start + batch_size,
                len(X_val)
            )

            X_batch = np.array(
                X_val[start:end],
                dtype=np.float32,
                copy=True
            )

            y_batch = np.array(
                y_val[start:end],
                dtype=np.float32,
                copy=True
            )

            X_tensor = torch.from_numpy(
                X_batch
            ).to(
                DEVICE,
                non_blocking=True
            )

            logits = model(X_tensor).squeeze(1)

            probs = torch.sigmoid(logits)

            preds = (
                probs >= 0.5
            ).cpu().numpy().astype(np.int32)

            predictions.extend(preds)
            actual.extend(
                y_batch.astype(np.int32)
            )

    accuracy = accuracy_score(
        actual,
        predictions
    )

    precision = precision_score(
        actual,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        actual,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        actual,
        predictions,
        zero_division=0
    )

    print(
        f"Round {server_round} - "
        f"Accuracy: {accuracy:.4f}, "
        f"Precision: {precision:.4f}, "
        f"Recall: {recall:.4f}, "
        f"F1: {f1:.4f}"
    )

    return 0.0, {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# Capture the final aggregated global parameters
final_parameters = None


class SaveFinalParametersStrategy(
    fl.server.strategy.FedAvg
):

    def aggregate_fit(
        self,
        server_round,
        results,
        failures
    ):
        global final_parameters

        aggregated_parameters, aggregated_metrics = (
            super().aggregate_fit(
                server_round,
                results,
                failures
            )
        )

        if aggregated_parameters is not None:
            final_parameters = aggregated_parameters

        return (
            aggregated_parameters,
            aggregated_metrics
        )


strategy = SaveFinalParametersStrategy(
    fraction_fit=1.0,
    fraction_evaluate=0.0,
    min_fit_clients=4,
    min_available_clients=4,
    evaluate_fn=evaluate_global_model
)

print(
    "FedAvg strategy with final model capture "
    "configured successfully."
)

FedAvg strategy with final model capture configured successfully.


In [ ]:
# File: Colab notebook cell — Flower federated simulation

NUM_ROUNDS = 3

print("Starting Flower federated simulation...")
print(f"Clients: 4")
print(f"Rounds: {NUM_ROUNDS}")
print(f"Device: {DEVICE}")
print("GPU clients will be scheduled one at a time.")


history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=4,
    config=fl.server.ServerConfig(
        num_rounds=NUM_ROUNDS
    ),
    strategy=strategy,
    client_resources={
        "num_cpus": 1,
        "num_gpus": 1
    }
)

print("\nFederated simulation completed.")

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=3, no round_timeout


Starting Flower federated simulation...
Clients: 4
Rounds: 3
Device: cuda
GPU clients will be scheduled one at a time.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-09-22 14:57:37,201	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'memory': 9299430605.0, 'object_store_memory': 3985470259.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'GPU': 1.0, 'node:172.28.0.12': 1.0, 'CPU': 2.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 1}
INFO :      Flower VCE: Creating VirtualClientEngineActorPool with 1 actors
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
INFO :     

Round 0 - Accuracy: 0.8026, Precision: 0.0000, Recall: 0.0000, F1: 0.0000
(ClientAppActor pid=18568) Epoch 1/1 - Loss: 0.0707
(ClientAppActor pid=18568) Epoch 1/1 - Loss: 0.0921
(ClientAppActor pid=18568) Epoch 1/1 - Loss: 0.1290


INFO :      aggregate_fit: received 4 results and 0 failures


...

INFO :      aggregate_fit: received 4 results and 0 failures


(ClientAppActor pid=18568) Epoch 1/1 - Loss: 0.0527


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Round 3 - Accuracy: 0.9753, Precision: 0.9692, Recall: 0.9031, F1: 0.9350

Federated simulation completed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# File: Colab notebook cell — Verify final FL parameters

print("Final parameters captured:", final_parameters is not None)

if final_parameters is not None:
    print(
        "Number of model parameter arrays:",
        len(final_parameters.tensors)
    )
else:
    print("ERROR: Final parameters were not captured.")

Final parameters captured: True
Number of model parameter arrays: 8


In [ ]:
# File: Colab notebook Cell 23 — Final FL test-set evaluation

from flwr.common import parameters_to_ndarrays

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# Convert Flower Parameters → NumPy arrays
final_weights = parameters_to_ndarrays(
    final_parameters
)

print(
    "Number of parameter arrays:",
    len(final_weights)
)


# Load untouched test data
X_test = np.load(
    DATA_DIR / "X_test.npy",
    mmap_mode="r"
)

y_test = np.load(
    DATA_DIR / "y_test.npy",
    mmap_mode="r"
)

print("Test samples:", len(X_test))


# Create final global model
final_model = IDSModel(input_size=78)

state_dict = final_model.state_dict()

new_state_dict = {
    key: torch.tensor(
        value,
        dtype=state_dict[key].dtype
    )
    for key, value in zip(
        state_dict.keys(),
        final_weights
    )
}

final_model.load_state_dict(
    new_state_dict
)

final_model.to(DEVICE)
final_model.eval()


# Test-set prediction
test_predictions = []
test_actual = []

batch_size = 4096

with torch.no_grad():

    for start in range(
        0,
        len(X_test),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(X_test)
        )

        X_batch = np.array(
            X_test[start:end],
            dtype=np.float32,
            copy=True
        )

        y_batch = np.array(
            y_test[start:end],
            dtype=np.float32,
            copy=True
        )

        X_tensor = torch.from_numpy(
            X_batch
        ).to(
            DEVICE,
            non_blocking=True
        )

        logits = final_model(
            X_tensor
        ).squeeze(1)

        probabilities = torch.sigmoid(
            logits
        )

        predictions = (
            probabilities >= 0.5
        ).cpu().numpy().astype(np.int32)

        test_predictions.extend(
            predictions
        )

        test_actual.extend(
            y_batch.astype(np.int32)
        )


# Calculate final metrics
test_accuracy = accuracy_score(
    test_actual,
    test_predictions
)

test_precision = precision_score(
    test_actual,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_actual,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_actual,
    test_predictions,
    zero_division=0
)


# Confusion matrix
cm = confusion_matrix(
    test_actual,
    test_predictions
)


print("\n" + "=" * 50)
print("FINAL FEDERATED LEARNING TEST RESULTS")
print("=" * 50)

print(f"Accuracy :  {test_accuracy:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall   :  {test_recall:.4f}")
print(f"F1-Score :  {test_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Number of parameter arrays: 8
Test samples: 283075


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


FINAL FEDERATED LEARNING TEST RESULTS
Accuracy :  0.9757
Precision:  0.9703
Recall   :  0.9046
F1-Score :  0.9363

Confusion Matrix:
[[225765   1545]
 [  5320  50445]]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# File: Colab notebook Cell 24 — Save final FL model and results

import json
import numpy as np
import torch


# Create output directories
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Save PyTorch model
model_path = MODEL_DIR / "fl_final_model.pt"

torch.save(
    final_model.state_dict(),
    model_path
)


# Save test metrics
metrics = {
    "experiment": "Federated Learning - Non-IID",
    "num_clients": 4,
    "num_rounds": 3,
    "local_epochs": 1,
    "accuracy": float(test_accuracy),
    "precision": float(test_precision),
    "recall": float(test_recall),
    "f1_score": float(test_f1)
}

metrics_path = RESULTS_DIR / "fl_test_metrics.json"

with open(
    metrics_path,
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=4
    )


# Save confusion matrix
cm_path = RESULTS_DIR / "fl_confusion_matrix.npy"

np.save(
    cm_path,
    cm
)


print("=" * 50)
print("FL EXPERIMENT SAVED")
print("=" * 50)

print("Model:", model_path)
print("Metrics:", metrics_path)
print("Confusion matrix:", cm_path)

print("\nFinal Test Metrics:")
print(f"Accuracy :  {test_accuracy:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall   :  {test_recall:.4f}")
print(f"F1-Score :  {test_f1:.4f}")

FL EXPERIMENT SAVED
Model: /content/IDSPRESERVING/models/fl_final_model.pt
Metrics: /content/IDSPRESERVING/results/fl_test_metrics.json
Confusion matrix: /content/IDSPRESERVING/results/fl_confusion_matrix.npy

Final Test Metrics:
Accuracy :  0.9757
Precision:  0.9703
Recall   :  0.9046
F1-Score :  0.9363


In [ ]:
# File: Colab notebook Cell 25 — Centralized training configuration

CENTRALIZED_BATCH_SIZE = 1024
CENTRALIZED_EPOCHS = 3
CENTRALIZED_LEARNING_RATE = 0.001

print("=" * 50)
print("CENTRALIZED DNN BASELINE")
print("=" * 50)

print("Training samples:", len(X_train) if "X_train" in globals() else "Will load")
print("Batch size:", CENTRALIZED_BATCH_SIZE)
print("Epochs:", CENTRALIZED_EPOCHS)
print("Learning rate:", CENTRALIZED_LEARNING_RATE)
print("Device:", DEVICE)

CENTRALIZED DNN BASELINE
Training samples: 2264594
Batch size: 1024
Epochs: 3
Learning rate: 0.001
Device: cuda


In [ ]:
# File: Colab notebook Cell 26 — Centralized training data loader

from torch.utils.data import Dataset, DataLoader


class CentralizedDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):

        x = np.array(
            self.X[index],
            dtype=np.float32,
            copy=True
        )

        y = np.float32(
            self.y[index]
        )

        return (
            torch.from_numpy(x),
            torch.tensor(
                y,
                dtype=torch.float32
            )
        )


# Load training data using memory mapping
X_train = np.load(
    DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train = np.load(
    DATA_DIR / "y_train.npy",
    mmap_mode="r"
)


# Create centralized dataset
centralized_dataset = CentralizedDataset(
    X_train,
    y_train
)


# Create DataLoader
centralized_loader = DataLoader(
    centralized_dataset,
    batch_size=CENTRALIZED_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)


print("=" * 50)
print("CENTRALIZED DATA LOADER")
print("=" * 50)

print("Training samples:", len(centralized_dataset))
print("Features:", X_train.shape[1])
print("Batch size:", CENTRALIZED_BATCH_SIZE)
print("Number of batches:", len(centralized_loader))

CENTRALIZED DATA LOADER
Training samples: 2264594
Features: 78
Batch size: 1024
Number of batches: 2212


In [ ]:
# File: Colab notebook Cell 27 — Centralized DNN training

centralized_model = IDSModel(
    input_size=78
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    centralized_model.parameters(),
    lr=CENTRALIZED_LEARNING_RATE
)


print("=" * 50)
print("STARTING CENTRALIZED TRAINING")
print("=" * 50)

centralized_model.train()

for epoch in range(CENTRALIZED_EPOCHS):

    total_loss = 0.0

    for batch_number, (X_batch, y_batch) in enumerate(
        centralized_loader,
        start=1
    ):

        X_batch = X_batch.to(
            DEVICE,
            non_blocking=True
        )

        y_batch = y_batch.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = centralized_model(
            X_batch
        ).squeeze(1)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_number % 200 == 0:
            print(
                f"Epoch {epoch + 1}/{CENTRALIZED_EPOCHS} | "
                f"Batch {batch_number}/{len(centralized_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = (
        total_loss /
        len(centralized_loader)
    )

    print(
        f"Epoch {epoch + 1}/{CENTRALIZED_EPOCHS} "
        f"completed | "
        f"Average Loss: {average_loss:.4f}"
    )

print("\nCentralized training completed.")

STARTING CENTRALIZED TRAINING
Epoch 1/3 | Batch 200/2212 | Loss: 0.1299
Epoch 1/3 | Batch 400/2212 | Loss: 0.0728
Epoch 1/3 | Batch 600/2212 | Loss: 0.0615
Epoch 1/3 | Batch 800/2212 | Loss: 0.0459
Epoch 1/3 | Batch 1000/2212 | Loss: 0.0603
Epoch 1/3 | Batch 1200/2212 | Loss: 0.0418
Epoch 1/3 | Batch 1400/2212 | Loss: 0.0606
Epoch 1/3 | Batch 1600/2212 | Loss: 0.0459
Epoch 1/3 | Batch 1800/2212 | Loss: 0.0693
Epoch 1/3 | Batch 2000/2212 | Loss: 0.0418
Epoch 1/3 | Batch 2200/2212 | Loss: 0.0498
Epoch 1/3 completed | Average Loss: 0.0714
Epoch 2/3 | Batch 200/2212 | Loss: 0.0506
Epoch 2/3 | Batch 400/2212 | Loss: 0.0498
Epoch 2/3 | Batch 600/2212 | Loss: 0.0502
Epoch 2/3 | Batch 800/2212 | Loss: 0.0630
Epoch 2/3 | Batch 1000/2212 | Loss: 0.0269
Epoch 2/3 | Batch 1200/2212 | Loss: 0.0349
Epoch 2/3 | Batch 1400/2212 | Loss: 0.0468
Epoch 2/3 | Batch 1600/2212 | Loss: 0.0328
Epoch 2/3 | Batch 1800/2212 | Loss: 0.0362
Epoch 2/3 | Batch 2000/2212 | Loss: 0.0604
Epoch 2/3 | Batch 2200/2212 | Lo

In [ ]:
# File: Colab notebook Cell 28 — Centralized DNN test-set evaluation

centralized_model.eval()

centralized_predictions = []
centralized_actual = []

batch_size = 4096

with torch.no_grad():

    for start in range(
        0,
        len(X_test),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(X_test)
        )

        X_batch = np.array(
            X_test[start:end],
            dtype=np.float32,
            copy=True
        )

        y_batch = np.array(
            y_test[start:end],
            dtype=np.float32,
            copy=True
        )

        X_tensor = torch.from_numpy(
            X_batch
        ).to(
            DEVICE,
            non_blocking=True
        )

        logits = centralized_model(
            X_tensor
        ).squeeze(1)

        probabilities = torch.sigmoid(
            logits
        )

        predictions = (
            probabilities >= 0.5
        ).cpu().numpy().astype(np.int32)

        centralized_predictions.extend(
            predictions
        )

        centralized_actual.extend(
            y_batch.astype(np.int32)
        )


# Calculate metrics
centralized_accuracy = accuracy_score(
    centralized_actual,
    centralized_predictions
)

centralized_precision = precision_score(
    centralized_actual,
    centralized_predictions,
    zero_division=0
)

centralized_recall = recall_score(
    centralized_actual,
    centralized_predictions,
    zero_division=0
)

centralized_f1 = f1_score(
    centralized_actual,
    centralized_predictions,
    zero_division=0
)


# Confusion matrix
centralized_cm = confusion_matrix(
    centralized_actual,
    centralized_predictions
)


print("=" * 50)
print("CENTRALIZED DNN TEST RESULTS")
print("=" * 50)

print(f"Accuracy :  {centralized_accuracy:.4f}")
print(f"Precision:  {centralized_precision:.4f}")
print(f"Recall   :  {centralized_recall:.4f}")
print(f"F1-Score :  {centralized_f1:.4f}")

print("\nConfusion Matrix:")
print(centralized_cm)

CENTRALIZED DNN TEST RESULTS
Accuracy :  0.9831
Precision:  0.9303
Recall   :  0.9881
F1-Score :  0.9583

Confusion Matrix:
[[223181   4129]
 [   666  55099]]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# File: Colab notebook Cell 29 — Save centralized experiment

import json
import numpy as np
import torch


# Directories
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Save centralized model
centralized_model_path = (
    MODEL_DIR / "centralized_final_model.pt"
)

torch.save(
    centralized_model.state_dict(),
    centralized_model_path
)


# Save centralized metrics
centralized_metrics = {
    "experiment": "Centralized DNN",
    "epochs": CENTRALIZED_EPOCHS,
    "batch_size": CENTRALIZED_BATCH_SIZE,
    "learning_rate": CENTRALIZED_LEARNING_RATE,
    "accuracy": float(centralized_accuracy),
    "precision": float(centralized_precision),
    "recall": float(centralized_recall),
    "f1_score": float(centralized_f1)
}

centralized_metrics_path = (
    RESULTS_DIR / "centralized_test_metrics.json"
)

with open(
    centralized_metrics_path,
    "w"
) as f:
    json.dump(
        centralized_metrics,
        f,
        indent=4
    )


# Save confusion matrix
centralized_cm_path = (
    RESULTS_DIR / "centralized_confusion_matrix.npy"
)

np.save(
    centralized_cm_path,
    centralized_cm
)


print("=" * 50)
print("CENTRALIZED EXPERIMENT SAVED")
print("=" * 50)

print("Model:", centralized_model_path)
print("Metrics:", centralized_metrics_path)
print("Confusion matrix:", centralized_cm_path)

print("\nFinal Test Metrics:")
print(f"Accuracy :  {centralized_accuracy:.4f}")
print(f"Precision:  {centralized_precision:.4f}")
print(f"Recall   :  {centralized_recall:.4f}")
print(f"F1-Score :  {centralized_f1:.4f}")

CENTRALIZED EXPERIMENT SAVED
Model: /content/IDSPRESERVING/models/centralized_final_model.pt
Metrics: /content/IDSPRESERVING/results/centralized_test_metrics.json
Confusion matrix: /content/IDSPRESERVING/results/centralized_confusion_matrix.npy

Final Test Metrics:
Accuracy :  0.9831
Precision:  0.9303
Recall   :  0.9881
F1-Score :  0.9583


In [ ]:
# File: Colab notebook Cell 30 — Final experiment comparison

import json
import pandas as pd


# Create comparison table
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ],
    "Centralized DNN": [
        centralized_accuracy,
        centralized_precision,
        centralized_recall,
        centralized_f1
    ],
    "Non-IID Federated DNN": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ]
})


# Convert to percentages
comparison["Centralized DNN"] *= 100
comparison["Non-IID Federated DNN"] *= 100


# Display
print("=" * 65)
print("CENTRALIZED DNN vs NON-IID FEDERATED DNN")
print("=" * 65)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}%"
    )
)


# Save CSV
comparison_csv_path = (
    RESULTS_DIR / "centralized_vs_federated_comparison.csv"
)

comparison.to_csv(
    comparison_csv_path,
    index=False
)


# Save JSON
comparison_json = {
    "centralized_dnn": {
        "accuracy": float(centralized_accuracy),
        "precision": float(centralized_precision),
        "recall": float(centralized_recall),
        "f1_score": float(centralized_f1)
    },
    "non_iid_federated_dnn": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1)
    }
}

comparison_json_path = (
    RESULTS_DIR / "centralized_vs_federated_comparison.json"
)

with open(
    comparison_json_path,
    "w"
) as f:
    json.dump(
        comparison_json,
        f,
        indent=4
    )


print("\nSaved files:")
print(comparison_csv_path)
print(comparison_json_path)

CENTRALIZED DNN vs NON-IID FEDERATED DNN
   Metric  Centralized DNN  Non-IID Federated DNN
 Accuracy           98.31%                 97.57%
Precision           93.03%                 97.03%
   Recall           98.81%                 90.46%
 F1-Score           95.83%                 93.63%

Saved files:
/content/IDSPRESERVING/results/centralized_vs_federated_comparison.csv
/content/IDSPRESERVING/results/centralized_vs_federated_comparison.json


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
